<a href="https://colab.research.google.com/github/kananmmehta/Reinforcement-Learning/blob/main/Liar's_Dice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import math
from typing import List, Tuple, Dict

# ==========================================
# 1. CORE MATH & PROBABILITY FUNCTIONS
# ==========================================

def binomial_probability(n: int, k: int, p: float) -> float:
    """Calculates the probability of getting at least k successes in n trials."""
    if k <= 0:
        return 1.0
    if k > n:
        return 0.0

    prob_exact_or_more = 0.0
    for i in range(k, n + 1):
        # nCr * p^i * (1-p)^(n-i)
        n_choose_i = math.comb(n, i)
        prob_exact_or_more += n_choose_i * (p ** i) * ((1 - p) ** (n - i))
    return prob_exact_or_more

def evaluate_bid_probability(bid_quantity: int, bid_value: int, my_dice: List[int], total_dice_in_play: int) -> float:
    """
    Calculates the probability that the bid is valid, given the AI's own dice.
    Accounts for '1' being wild.
    """
    # Count how many matching dice the AI already holds
    my_matching = my_dice.count(bid_value)
    if bid_value != 1:
        my_matching += my_dice.count(1) # Add wild cards
        p_success = 2 / 6 # Normal face value + Wild 1
    else:
        p_success = 1 / 6 # Just 1s

    needed = max(0, bid_quantity - my_matching)
    unknown_dice = total_dice_in_play - len(my_dice)

    return binomial_probability(unknown_dice, needed, p_success)

# ==========================================
# 2. AI AGENT IMPLEMENTATION
# ==========================================

class LiarsDiceAI:
    def __init__(self, name: str, bluff_factor: float = 0.15):
        self.name = name
        self.bluff_factor = bluff_factor # Likelihood to bid slightly higher than mathematically safe
        self.dice = []

    def roll_dice(self, num_dice: int):
        self.dice = sorted([random.randint(1, 6) for _ in range(num_dice)])

    def make_move(self, current_bid: Tuple[int, int], total_dice: int) -> Tuple[str, Tuple[int, int]]:
        """
        Decides whether to challenge ("Liar") or raise ("Bid").
        Returns: ('Liar', None) OR ('Bid', (new_quantity, new_value))
        """
        # If it's the opening turn
        if current_bid == (0, 0):
            # Bid based on our most common die face
            counts = {face: self.dice.count(face) + (0 if face == 1 else self.dice.count(1)) for face in range(1, 7)}
            best_face = max(counts, key=counts.get)
            # Safe opening: bid your own count, minimum 1
            opening_qty = max(1, counts[best_face])
            return "Bid", (opening_qty, best_face)

        curr_qty, curr_val = current_bid

        # 1. Evaluate math probability of current bid being true
        prob = evaluate_bid_probability(curr_qty, curr_val, self.dice, total_dice)

        # 2. Determine threshold to call "Liar"
        # We add a slight random variance to mimic human psychology/bluffing
        liar_threshold = 0.25 + random.uniform(-0.05, 0.05)

        if prob < liar_threshold:
            return "Liar", (0, 0)

        # 3. Formulate a counter-bid (Escalation)
        # Find the best face value for the AI based on its hand
        counts = {face: self.dice.count(face) + (0 if face == 1 else self.dice.count(1)) for face in range(1, 7)}

        # We look for a valid raise option
        best_bid = None
        min_probability_loss = -1

        # Test valid bid combinations to find the one with the highest mathematical safety
        for face in range(1, 7):
            for qty in range(curr_qty, total_dice + 1):
                # Enforce Liar's Dice standard bidding rule:
                # Either quantity must be higher, or if quantity is same, face value must be higher.
                if qty > curr_qty or (qty == curr_qty and face > curr_val):
                    bid_prob = evaluate_bid_probability(qty, face, self.dice, total_dice)

                    # Apply a minor bluff boost to AI's favorite faces
                    if face == max(counts, key=counts.get):
                        bid_prob += self.bluff_factor

                    if bid_prob > min_probability_loss:
                        min_probability_loss = bid_prob
                        best_bid = (qty, face)

        # If no safe bid is found or probability is too low, default to calling Liar
        if best_bid is None or min_probability_loss < 0.15:
            return "Liar", (0, 0)

        return "Bid", best_bid

# ==========================================
# 3. GAME ENVIRONMENT ENGINE
# ==========================================

class LiarsDiceGame:
    def __init__(self, human_name: str, ai_names: List[str], dice_per_player: int = 5):
        self.human_name = human_name
        self.ais = [LiarsDiceAI(name) for name in ai_names]
        self.players = [human_name] + [ai.name for ai in self.ais]

        # Track dice counts
        self.dice_counts = {name: dice_per_player for name in self.players}
        self.human_dice = []
        self.current_bid = (0, 0) # (quantity, value)
        self.current_turn_idx = 0

    def get_total_dice(self) -> int:
        return sum(self.dice_counts.values())

    def start_round(self):
        print("\n=== 🎲 ROLLING DICE FOR A NEW ROUND 🎲 ===")
        self.current_bid = (0, 0)

        # Roll AI Dice
        for ai in self.ais:
            if self.dice_counts[ai.name] > 0:
                ai.roll_dice(self.dice_counts[ai.name])

        # Roll Human Dice
        if self.dice_counts[self.human_name] > 0:
            self.human_dice = sorted([random.randint(1, 6) for _ in range(self.dice_counts[self.human_name])])
            print(f"Your Hand ({self.human_name}): {self.human_dice}")

        print(f"Total dice remaining in play: {self.get_total_dice()}")
        print("------------------------------------------")

    def count_all_dice(self, value: int) -> int:
        """Counts the total actual dice across all players matching the value (including wild 1s)"""
        total = 0
        # Count human
        total += self.human_dice.count(value)
        if value != 1:
            total += self.human_dice.count(1)

        # Count AIs
        for ai in self.ais:
            if self.dice_counts[ai.name] > 0:
                total += ai.dice.count(value)
                if value != 1:
                    total += ai.dice.count(1)
        return total

    def play_game(self):
        while len([p for p in self.players if self.dice_counts[p] > 0]) > 1:
            self.start_round()

            # Find the first active player for the round
            while self.dice_counts[self.players[self.current_turn_idx]] == 0:
                self.current_turn_idx = (self.current_turn_idx + 1) % len(self.players)

            last_bidder = None
            round_active = True

            while round_active:
                current_player = self.players[self.current_turn_idx]

                # Skip eliminated players
                if self.dice_counts[current_player] == 0:
                    self.current_turn_idx = (self.current_turn_idx + 1) % len(self.players)
                    continue

                print(f"\nCurrent Bid: {f'{self.current_bid[0]}x [🎲 {self.current_bid[1]}]' if self.current_bid != (0,0) else 'None'}")
                print(f"👉 It's {current_player}'s turn!")

                # --- HUMAN TURN ---
                if current_player == self.human_name:
                    print(f"Your dice: {self.human_dice}")
                    action = input("Choose action: [B]id or [L]iar: ").strip().upper()

                    if action == 'L' and self.current_bid != (0, 0):
                        round_active = False
                        decision = "Liar"
                    else:
                        decision = "Bid"
                        valid_input = False
                        while not valid_input:
                            try:
                                qty = int(input("Enter Quantity: "))
                                val = int(input("Enter Face Value (1-6): "))
                                # Rule check
                                if val < 1 or val > 6:
                                    print("Face value must be between 1 and 6.")
                                elif qty > self.get_total_dice():
                                    print(f"You can't bid more than the total dice in play ({self.get_total_dice()})!")
                                elif qty > self.current_bid[0] or (qty == self.current_bid[0] and val > self.current_bid[1]):
                                    self.current_bid = (qty, val)
                                    valid_input = True
                                else:
                                    print("Invalid raise! You must increase the quantity, or increase the face value for the same quantity.")
                            except ValueError:
                                print("Please enter valid integers.")

                # --- AI TURN ---
                else:
                    ai_agent = next(ai for ai in self.ais if ai.name == current_player)
                    decision, new_bid = ai_agent.make_move(self.current_bid, self.get_total_dice())

                    if decision == "Liar":
                        print(f"🤖 {current_player} screams: \"LIAR!!!\"")
                        round_active = False
                    else:
                        self.current_bid = new_bid
                        print(f"🤖 {current_player} bids: {self.current_bid[0]}x [🎲 {self.current_bid[1]}]")

                # Resolution logic if "Liar" is called
                if not round_active:
                    print("\n--- 📣 SHOWDOWN 📣 ---")
                    actual_count = self.count_all_dice(self.current_bid[1])

                    # Print what everyone had
                    if self.dice_counts[self.human_name] > 0:
                        print(f"{self.human_name} had: {self.human_dice}")
                    for ai in self.ais:
                        if self.dice_counts[ai.name] > 0:
                            print(f"{ai.name} had: {ai.dice}")

                    print(f"\nTarget Bid: {self.current_bid[0]}x [🎲 {self.current_bid[1]}]")
                    print(f"Actual Count (including wild 1s): {actual_count}")

                    # Determine who loses a die
                    if actual_count >= self.current_bid[0]:
                        # Bid was true! The challenger loses a die.
                        loser = current_player
                        print(f"🏆 The bid was TRUE! Challenger {loser} loses 1 die.")
                    else:
                        # Bid was a lie! The bidder loses a die.
                        loser = last_bidder
                        print(f"🏆 The bid was a LIE! Bidder {loser} loses 1 die.")

                    self.dice_counts[loser] -= 1
                    print(f"\nRemaining Dice Stats: {self.dice_counts}")

                    if self.dice_counts[loser] == 0:
                        print(f"❌ {loser} HAS BEEN ELIMINATED FROM THE GAME!")

                    # Loser starts the next round if still alive
                    if self.dice_counts[loser] > 0:
                        self.current_turn_idx = self.players.index(loser)
                    break

                # Pass turn to next player
                last_bidder = current_player
                self.current_turn_idx = (self.current_turn_idx + 1) % len(self.players)

        winner = [p for p in self.players if self.dice_counts[p] > 0][0]
        print(f"\n🏆🎉 GAME OVER! THE ULTIMATE WINNER IS {winner}! 🎉🏆")

# ==========================================
# 4. RUN THE GAME
# ==========================================
# Instantiating a game with You vs 2 AI Agents
game = LiarsDiceGame(human_name="You", ai_names=["DeepDice-Bot", "BluffMaster-9000"])
game.play_game()


=== 🎲 ROLLING DICE FOR A NEW ROUND 🎲 ===
Your Hand (You): [1, 3, 4, 5, 6]
Total dice remaining in play: 15
------------------------------------------

Current Bid: None
👉 It's You's turn!
Your dice: [1, 3, 4, 5, 6]
Choose action: [B]id or [L]iar: B
Enter Quantity: 2
Enter Face Value (1-6): 3

Current Bid: 2x [🎲 3]
👉 It's DeepDice-Bot's turn!
🤖 DeepDice-Bot bids: 2x [🎲 4]

Current Bid: 2x [🎲 4]
👉 It's BluffMaster-9000's turn!
🤖 BluffMaster-9000 bids: 3x [🎲 2]

Current Bid: 3x [🎲 2]
👉 It's You's turn!
Your dice: [1, 3, 4, 5, 6]
Choose action: [B]id or [L]iar: L

--- 📣 SHOWDOWN 📣 ---
You had: [1, 3, 4, 5, 6]
DeepDice-Bot had: [1, 3, 4, 4, 5]
BluffMaster-9000 had: [1, 2, 2, 4, 5]

Target Bid: 3x [🎲 2]
Actual Count (including wild 1s): 5
🏆 The bid was TRUE! Challenger You loses 1 die.

Remaining Dice Stats: {'You': 4, 'DeepDice-Bot': 5, 'BluffMaster-9000': 5}

=== 🎲 ROLLING DICE FOR A NEW ROUND 🎲 ===
Your Hand (You): [1, 3, 4, 5]
Total dice remaining in play: 14
---------------------------